# Transmission Switching - In-Class Example

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Two-bus, three-parallel-line system. Line status variables decide which lines are in service.

In [1]:
from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, SolverFactory,
    Binary, minimize, value
)

MVABase = 100.0
x1, x2, x3 = 0.01, 0.02, 0.02
lineLimit1, lineLimit2, lineLimit3 = 50.0, 20.0, 10.0
LoadA, LoadB = 0.0, 150.0
Gamin, Gamax = 0.0, 200.0
Gbmin, Gbmax = 0.0, 100.0
CostGa, CostGb = 10.0, 30.0
BigM = 10e7

m = ConcreteModel()

m.lineflow1 = Var()
m.lineflow2 = Var()
m.lineflow3 = Var()
m.line_status1 = Var(domain=Binary)
m.line_status2 = Var(domain=Binary)
m.line_status3 = Var(domain=Binary)
m.thetaA = Var()
m.thetaB = Var()
m.Ga = Var(bounds=(Gamin/MVABase, Gamax/MVABase))
m.Gb = Var(bounds=(Gbmin/MVABase, Gbmax/MVABase))

m.totalCost = Objective(expr=CostGa*m.Ga*MVABase + CostGb*m.Gb*MVABase, sense=minimize)

m.NodalA = Constraint(expr=m.Ga - LoadA/MVABase - m.lineflow1 - m.lineflow2 - m.lineflow3 == 0)
m.NodalB = Constraint(expr=m.Gb - LoadB/MVABase + m.lineflow1 + m.lineflow2 + m.lineflow3 == 0)

# Big-M reformulation of the bilinear line-flow constraints
m.LF1_1 = Constraint(expr=-BigM*(1 - m.line_status1) <= m.lineflow1 - (m.thetaA - m.thetaB)/x1)
m.LF1_2 = Constraint(expr= m.lineflow1 - (m.thetaA - m.thetaB)/x1 <= BigM*(1 - m.line_status1))
m.LF2_1 = Constraint(expr=-BigM*(1 - m.line_status2) <= m.lineflow2 - (m.thetaA - m.thetaB)/x2)
m.LF2_2 = Constraint(expr= m.lineflow2 - (m.thetaA - m.thetaB)/x2 <= BigM*(1 - m.line_status2))
m.LF3_1 = Constraint(expr=-BigM*(1 - m.line_status3) <= m.lineflow3 - (m.thetaA - m.thetaB)/x3)
m.LF3_2 = Constraint(expr= m.lineflow3 - (m.thetaA - m.thetaB)/x3 <= BigM*(1 - m.line_status3))

m.LL1_1 = Constraint(expr=-m.line_status1*lineLimit1/MVABase <= m.lineflow1)
m.LL1_2 = Constraint(expr= m.lineflow1 <=  m.line_status1*lineLimit1/MVABase)
m.LL2_1 = Constraint(expr=-m.line_status2*lineLimit2/MVABase <= m.lineflow2)
m.LL2_2 = Constraint(expr= m.lineflow2 <=  m.line_status2*lineLimit2/MVABase)
m.LL3_1 = Constraint(expr=-m.line_status3*lineLimit3/MVABase <= m.lineflow3)
m.LL3_2 = Constraint(expr= m.lineflow3 <=  m.line_status3*lineLimit3/MVABase)

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)

m = model

# ---- AMPL-style output (matches slide "2-Bus Example") ----
print(f"line_status1 = {int(round(value(m.line_status1)))}")
print(f"line_status2 = {int(round(value(m.line_status2)))}")
print(f"line_status3 = {int(round(value(m.line_status3)))}")
print()
print(f"lineflow1*MVABase = {value(m.lineflow1)*MVABase:.4g}")
print(f"lineflow2*MVABase = {value(m.lineflow2)*MVABase:.4g}")
print(f"lineflow3*MVABase = {value(m.lineflow3)*MVABase:.4g}")
print()
print(f"Ga*MVABase = {value(m.Ga)*MVABase:.4g}")
print(f"Gb*MVABase = {value(m.Gb)*MVABase:.4g}")


Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpk804200j.pyomo.lp


Reading time = 0.00 seconds
x1: 14 rows, 10 columns, 44 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90


MIPGap  0

Optimize a model with 14 rows, 10 columns and 44 nonzeros


Model fingerprint: 0x0ed09fbf
Variable types: 7 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+08]
  Objective range  [1e+03, 3e+03]
  Bounds range     [1e+00, 2e+00]
  RHS range        [2e+00, 1e+08]


Presolve removed 10 rows and 7 columns


Presolve time: 0.00s
Presolved: 4 rows, 3 columns, 10 nonzeros


Variable types: 2 continuous, 1 integer (1 binary)
Found heuristic solution: objective 3299.9999762



Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 20 (of 20 available processors)

Solution count 1: 3300 

Optimal solution found (tolerance 0.00e+00)


Best objective 3.299999976158e+03, best bound 3.299999976158e+03, gap 0.0000%


ok optimal
line_status1 = 1
line_status2 = 1
line_status3 = 0

lineflow1*MVABase = 40
lineflow2*MVABase = 20
lineflow3*MVABase = 0

Ga*MVABase = 60
Gb*MVABase = 90
